# Rerunning detection and tracking on the dev scenes

SAM 3 and tracking again, on a GPU, for dev scenes (no question of `explore200_ids.txt`), on the 16 frames their
sheets were made on, with other detection settings. The logic is in `rerun_detection.py` and `track_many.py`; this
notebook runs them.

- **SAM 3**, `facebook/sam3` at revision `3c879f3`, the transformers backend, as `sam3_infer.py` runs it (its own
  functions): once per scene, for every SAM 3 prompt the chosen runs need, keeping **every detection down to score
  0.2** with its score. It is saved on Drive and reused by every later run; a prompt a scene's saved run lacks (a run
  chosen later, or a change to the prompt fixes) is run on its own and added to it, so no prompt is segmented twice.
- **Tracking**, `openvocab-4D-main/trackA/milestone4_video.py`'s track stage on those masks, with re-identification
  (DINOv2), for each chosen run and scene; several scenes of a run share one process (`track_many.py`).

**The plan (step 1)** is staged to keep the GPU time low: 15 scenes (`dev_scenes_15.txt`), 5 per dataset drawn with a
fixed seed, ScanNet++'s 3 from the lower half of its scenes by recall and 2 from the upper half (`python
rerun_detection.py` makes the file again; `dev_scenes.txt`, all 50, stays available as `SCENES_FILE`):

| Run | Prompts | Tracking | Scenes | Compared with |
|---|---|---|---|---|
| `fixed_s020` | fixed | score ≥ 0.2 | 15 | `original_s020`: what the prompt fixes add |
| `original_s020` | original | score ≥ 0.2 | 15 | the saved sheets: what the lower threshold adds |
| `original_current` | original | the pipeline's: score ≥ 0.55, matching within 5% of the scene's diagonal, re-id similarity 0.6 within 15% | 3 (`CHECK_SCENES`) | the saved tracks: the reproduction check |
| `fixed_current`, `original_near10`, `original_reid_loose` | | the pipeline's; matching within 10%; re-id similarity 0.5 within 25% | off | add them to `RUNS` later |

The reproduction scenes are one per dataset: of the subset's scenes with one saved SAM 3 run, the one with the fewest
SAM 3 prompts.

**Fixed prompts** change only what the question's own text shows: two prompts the question writes next to each other
become one name ("ceiling" + "light" → "ceiling light"); a prompt the question writes only in the plural is named as
the question writes it and SAM 3 is prompted with both forms ("shoe" → "shoes", in `scannetpp/d755b3d9d8` of the
subset); and SAM 3 is also prompted with other common names (`SYNONYMS`: "trash bin" with "trash can" and "garbage
bin", "sofa" with "couch", ...), the detections pooled under the question's name, a detection dropped if it overlaps
(IoU ≥ 0.5) a higher-scored one of another of the name's prompts. Nothing is tuned on scenes outside `dev_scenes.txt`:
only a frozen configuration (`rerun_detection.FROZEN`, e.g. `pipeline_v3`) runs on other scenes, such as explore200's
(`explore200_scenes.txt`); the notebook refuses any other run there.

The tracker keeps detections from a score of 0.55 and matches within 5% of the scene's diagonal, both constants in its
code. Runs with other values use a copy of it with those two constants made options (`rerun_detection.patch_tracker`),
otherwise the same code; runs with the pipeline's values use it unchanged.

**Files on Google Drive**, in `MyDrive/mvp/`:

| File | What |
|---|---|
| `mvp-3d-vlm-rerun.zip` | the code: `git archive` of `scene_sheet_formats` at `7087890` (`src/`, `openvocab-4D-main/`, `dev_scenes.txt`) plus `rerun_detection.py`, `track_many.py` and `dev_scenes_15.txt` |
| `rerun_inputs_explore200.zip` (only for frozen runs on explore200's scenes) | the same for the 25 scenes of `explore200_scenes.txt`, and that list |
| `rerun_inputs_dev.zip` | per dev scene (all 50), from `data/scene_sheets_75`: the 16 frames, `cameras.npz`, `adapter_report.json`, the saved SAM 3 masks and tracks, every sheet's `question.json`, the gold sheets; `depth.npz`, the 16 frames cut from `depth_vsibench_75.zip`'s dense depth as `rebuild_scene_sheets.ensure_depth` cuts it; `index.csv`, `dev_scenes.txt` (318 MB) |

**Hugging Face access.** `facebook/sam3` is gated (manually approved). With the account whose token you use, request
access on https://huggingface.co/facebook/sam3 (accept Meta's SAM license there) and wait for approval; then add a
token with read access as the Colab secret `HF_TOKEN` (the key icon on the left, with notebook access on). Step 3
checks the access before loading anything; it is skipped when every scene's SAM 3 run is already on Drive.

**Outputs**, in `MyDrive/mvp/rerun_detection/`:

- `sam3/<dataset>__<scene>/`: the SAM 3 run, `<frame>__masks.npz` and `manifest.json` (`sam3_infer.py`'s format, from
  score 0.2)
- `runs/<run>/`: a sheets folder like `data/scene_sheets_75` for the run's scenes, without the sheets' rows:
  `index.csv` and `scenes.txt` (the run's scenes, prompts as the run uses them), per scene
  `subset-uniform16/masks/rerun_<prompts>/`, `subset-uniform16/scene/tracks/<prompt>.npz` (+ `.settings.json`),
  `cameras.npz`, `adapter_report.json`, each sheet's `question.json`, and `DONE.json`, written last
- `runs.json` (the plan, the runs' settings, the synonyms, the code's fingerprints), `logs/` (tracking logs,
  `timings.jsonl`)

Every scene is written to Drive as soon as it is finished, under a temporary name and then renamed, and a finished
scene (its `manifest.json` with every prompt, or `DONE.json`) is skipped, so after a disconnect, run everything again
and it continues.

**Order:** run steps 1–5 (the 3 reproduction scenes: every chosen run on them, four checks, and the time estimate
measured on this GPU), read the checks, then set `ALL_SCENES = True` in step 1 and run step 6.

**batch2_A** (`prereg_batch2_A.md`, step 3): this copy runs the frozen `pipeline_v3` (SAM 3 and tracking) on the 38
scenes of `batch2_A_scenes.txt`, from `rerun_inputs_batch2_A.zip` (made by `batch2_A_inputs.ipynb`), into
`MyDrive/mvp/rerun_detection_batch2_A/`. Only the settings cell (step 1) differs from `rerun_detection.ipynb`, and step
5 (the reproduction checks against the dev scenes' saved masks) is left out. Run every cell from the top; after a
disconnect, run them all again: finished scenes are skipped.


## 1. Settings

In [ ]:
USE_DRIVE = True                          # False: put the two zips in /content yourself and set DRIVE_DIR to "/content"
DRIVE_DIR = "/content/drive/MyDrive/mvp"
CODE_ZIP = f"{DRIVE_DIR}/mvp-3d-vlm-rerun-batch2.zip"
INPUTS_ZIP = f"{DRIVE_DIR}/rerun_inputs_batch2_A.zip"   # written by batch2_A_inputs.ipynb
EXTRA_INPUTS_ZIP = ""                      # none: explore200's inputs must not be unpacked over batch2_A's index.csv
OUT_DIR = f"{DRIVE_DIR}/rerun_detection_batch2_A"   # its own folder, so the dev and explore200 runs' index.csv stay
SCENES_FILE = "batch2_A_scenes.txt"        # the 38 scenes of batch2_A (prereg_batch2_A.md)
RUNS = {"pipeline_v3": "all"}              # frozen (rerun_detection.FROZEN): fixed2_s040's prompts and tracking, with track scores
CHECK_SCENES = []                          # step 5 is left out: its checks compare with the dev scenes' saved masks
SCENES_PER_PROCESS = 5                     # scenes of a run tracked in one process; 1: a process per scene
ALL_SCENES = True                          # step 6 runs every scene


## 2. Files and packages

In [ ]:
import json, math, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
for f in (CODE_ZIP, INPUTS_ZIP):
    assert Path(f).is_file(), f"missing {f}: upload it to {DRIVE_DIR} (see Files on Google Drive above)"
CODE, INPUTS, WORK = Path("/content/mvp-3d-vlm"), Path("/content/rerun_inputs"), Path("/content/work")
OUT, LOGS = Path(OUT_DIR), Path(OUT_DIR) / "logs"
for z, target in ((CODE_ZIP, CODE), (INPUTS_ZIP, INPUTS)):
    if not target.is_dir():
        zipfile.ZipFile(z).extractall("/content")
if Path(EXTRA_INPUTS_ZIP).is_file() and not (INPUTS / "explore200_scenes.txt").is_file():
    zipfile.ZipFile(EXTRA_INPUTS_ZIP).extractall("/content")  # into /content/rerun_inputs too
LOGS.mkdir(parents=True, exist_ok=True)

# the cluster environment's versions (src/mvp/openvocab_adapter/hpc/setup_env.sh): transformers 5.20.1 for Sam3Model,
# and VGGT at OpenVocab-4D's pinned commit, which the tracker imports its geometry from (no VGGT weights are used)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.20.1"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                "git+https://github.com/facebookresearch/vggt@a288dd0f14786c93483e45524328726ab7b1b4ce"], check=True)
import torch
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > A100 or L4"
GPU = torch.cuda.get_device_name(0)
print("GPU:", GPU)
from transformers import Sam3Model  # noqa: F401  fails here if this transformers has no SAM 3

sys.path[:0] = [str(CODE / "experiments/2026-11-18-scene-sheet-formats"), str(CODE / "src/mvp/openvocab_adapter")]
import rerun_detection as rd
import sam3_infer

## 3. Plan, a first time estimate, Hugging Face access

The first estimate, before anything is measured, scales the cluster's B200 runs of the same SAM 3 setup (the saved
masks' `manifest.json`: about 20 s to load the model, then per scene about 8 s for the 16 frames and 1.3 s per prompt)
by an assumed factor per GPU (A100 2.5×, L4 5× slower), plus about 60 s to download the weights; tracking about 8 s
per scene at the pipeline's score and 20 s at 0.2 (more detections), 20% more on an L4, and 15 s to start each tracker
process (Python, PyTorch, DINOv2). Step 5 measures it on this GPU.

In [ ]:
DEV = (INPUTS / "dev_scenes.txt").read_text().split()
SCENES = (CODE / "experiments/2026-11-18-scene-sheet-formats" / SCENES_FILE).read_text().split()
assert set(RUNS) <= set(rd.RUNS), f"unknown runs: {set(RUNS) - set(rd.RUNS)}"
PLAN = {run: list(SCENES) if which == "all" else list(CHECK_SCENES) if which == "check" else list(which)
        for run, which in RUNS.items()}
# settings are chosen on the dev scenes only: a run on any other scene must be a frozen configuration (rd.FROZEN)
outside = {r: sorted(set(v) - set(DEV)) for r, v in PLAN.items() if set(v) - set(DEV)}
assert all(r in rd.FROZEN for r in outside), f"only frozen configurations run outside the dev scenes: {sorted(set(outside) - set(rd.FROZEN))}"
assert "check" not in RUNS.values() or set(CHECK_SCENES) <= set(SCENES), f"CHECK_SCENES not in {SCENES_FILE}"
PLAN_SCENES = sorted({s for v in PLAN.values() for s in v})
no_inputs = [s for s in PLAN_SCENES if not (rd.scene_dir(INPUTS, s) / rd.LABEL / "scene/depth.npz").is_file()]
assert not no_inputs, f"no inputs for {len(no_inputs)} scenes, e.g. {no_inputs[:3]}: upload their inputs zip (Files on Google Drive)"
INDEX = rd.read_index(INPUTS / "index.csv", set(DEV) | set(PLAN_SCENES))
BY_SCENE = {s: [r for r in INDEX if r["scene"] == s] for s in set(DEV) | set(PLAN_SCENES)}
# SAM 3 runs for the chosen runs' prompts; a run chosen later that needs other prompts has them added to the scene's
# SAM 3 run (ensure_sam3), so no scene's SAM 3 runs twice for a prompt
SAM_PROMPTS = {s: rd.sam_prompts(BY_SCENE[s], list(RUNS)) for s in PLAN_SCENES}
NAME = {s: s.replace("/", "__") for s in BY_SCENE}  # every scene the plan may name


def sam_missing(scene):
    return rd.missing_prompts(OUT / "sam3" / NAME[scene], SAM_PROMPTS[scene])


def done(run, scene):
    return (OUT / "runs" / run / NAME[scene] / "DONE.json").is_file()


print(f"{len(PLAN_SCENES)} scenes; " + "; ".join(f"{r}: {len(v)} scenes" for r, v in PLAN.items()))
NEED_SAM = [s for s in PLAN_SCENES if sam_missing(s)]
print(f"SAM 3 to run for {len(NEED_SAM)} scenes, {sum(len(sam_missing(s)) for s in NEED_SAM)} prompts in all "
      f"(the others are on Drive)")

pending = {r: [s for s in v if not done(r, s)] for r, v in PLAN.items()}
for gpu, slower, track_more in (("A100", 2.5, 1.0), ("L4", 5.0, 1.2)):
    sam = (60 + 20 * slower if NEED_SAM else 0) + sum(slower * (8 + 1.33 * len(sam_missing(s))) for s in NEED_SAM)
    jobs = sum(len(v) * (20 if rd.settings(r)["det_score"] < 0.5 else 8) * track_more for r, v in pending.items())
    starts = 15 * sum(math.ceil(len(v) / SCENES_PER_PROCESS) for v in pending.values())
    print(f"first estimate on an {gpu}: SAM 3 {sam / 60:.0f} min, tracking {(jobs + starts) / 60:.0f} min "
          f"({starts / 60:.0f} of it starting processes), about {(sam + jobs + starts) / 60:.0f} min in all")

for run, scenes in PLAN.items():  # each run's index.csv and scenes.txt: its scenes' questions, its prompts
    rd.write_index([r for r in INDEX if r["scene"] in set(scenes)], run, OUT / "runs" / run / "index.csv")
    (OUT / "runs" / run / "scenes.txt").write_text("\n".join(scenes) + "\n")
(OUT / "runs.json").write_text(json.dumps({
    "plan": PLAN, "scenes_file": SCENES_FILE, "frozen": {r: rd.FROZEN[r] for r in RUNS if r in rd.FROZEN}, "check_scenes": CHECK_SCENES, "scenes_per_process": SCENES_PER_PROCESS,
    "runs": {r: rd.settings(r) for r in RUNS}, "synonyms": rd.SYNONYMS, "synonyms_dropped_in_fixed_v2": rd.SYNONYMS_DROPPED,
    "dedupe_iou": rd.DEDUPE_IOU,
    "sam3": {"model": "facebook/sam3", "revision": sam3_infer.SAM3_REV, "min_score": rd.SAVE_MIN_SCORE},
    "code": {f: rd.fingerprint(CODE / f) for f in ("experiments/2026-11-18-scene-sheet-formats/rerun_detection.py",
                                                    "experiments/2026-11-18-scene-sheet-formats/track_many.py",
                                                    str(rd.TRACKER), "src/mvp/openvocab_adapter/sam3_infer.py")},
    "gpu": GPU}, indent=1))

In [ ]:
if not NEED_SAM:
    print("every scene's SAM 3 run is on Drive: no Hugging Face access needed")
else:
    import huggingface_hub
    from google.colab import userdata
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception as error:
        raise SystemExit("add a Hugging Face token as the Colab secret HF_TOKEN (key icon, notebook access on)") from error
    try:
        huggingface_hub.hf_hub_download("facebook/sam3", "config.json", revision=sam3_infer.SAM3_REV)
    except huggingface_hub.errors.GatedRepoError as error:
        raise SystemExit("this token's account has no access to facebook/sam3 yet: request it on "
                         "https://huggingface.co/facebook/sam3 and wait for approval") from error
    print(f"facebook/sam3@{sam3_infer.SAM3_REV[:7]}: access ok")

## 4. SAM 3 and the tracker

In [ ]:
DETECT = MODEL_NAME = None
if NEED_SAM:
    t0 = time.time()
    DETECT, MODEL_NAME = rd.load_detector(CODE)  # downloads the weights (about 3.4 GB) the first time
    print(f"{MODEL_NAME} loaded in {time.time() - t0:.0f} s")
print("tracker copy with --det-score and --near-frac:", rd.patch_tracker(CODE).name)


def log_time(record):
    with open(LOGS / "timings.jsonl", "a") as f:
        f.write(json.dumps({"gpu": GPU, **record}) + "\n")


def raw_dir(scene):
    """The scene's SAM 3 run: this session's copy if it has every prompt, else Drive's."""
    local = WORK / "sam3" / NAME[scene]
    return local if not rd.missing_prompts(local, SAM_PROMPTS[scene]) else OUT / "sam3" / NAME[scene]


def ensure_sam3(scene):
    """SAM 3 for the prompts the scene's run on Drive lacks (all of them if there is none), added to it."""
    missing = sam_missing(scene)
    if not missing:
        return
    if DETECT is None:
        raise RuntimeError("SAM 3 is not loaded: run step 3 and 4 again")
    t0, final = time.time(), OUT / "sam3" / NAME[scene]
    frames = rd.frames_of(rd.scene_dir(INPUTS, scene))
    local = WORK / "sam3" / NAME[scene]
    shutil.rmtree(local, ignore_errors=True)
    if not (final / "manifest.json").is_file():
        rd.run_sam3(DETECT, MODEL_NAME, frames, missing, local)
    else:  # SAM 3 segments each prompt on its own: the added prompts' detections are those of one run with all
        extra = WORK / "sam3_added" / NAME[scene]
        shutil.rmtree(extra, ignore_errors=True)
        rd.run_sam3(DETECT, MODEL_NAME, frames, missing, extra)
        rd.merge_sam3(final, extra, local)
    rd.publish(local, final)
    log_time({"scene": scene, "sam3": time.time() - t0, "prompts": len(missing)})


def track_run(run, scenes):
    """The run's tracks for the scenes Drive doesn't have, SCENES_PER_PROCESS scenes per tracker process.
    Returns {scene: error} for the scenes that failed."""
    s, failed = rd.settings(run), {}
    todo = [sc for sc in scenes if not done(run, sc)]
    for i in range(0, len(todo), SCENES_PER_PROCESS):
        chunk = todo[i:i + SCENES_PER_PROCESS]
        work = WORK / "runs" / run
        jobs = [rd.prepare_run_scene(INPUTS, raw_dir(sc), run, BY_SCENE[sc], work) for sc in chunk]
        results, wall = rd.track_batch(CODE, jobs, s, WORK / "batch" / run, LOGS / f"{run}.log")
        for sc, r in zip(chunk, results):
            try:
                if not r["ok"]:
                    raise RuntimeError(r["error"])
                local = rd.finish_run_scene(INPUTS, run, BY_SCENE[sc], work)
                rd.publish(local, OUT / "runs" / run / NAME[sc])
                shutil.rmtree(local)
            except Exception as error:  # noqa: BLE001 - reported; the other scenes go on
                failed[sc] = f"{type(error).__name__}: {error}"
        log_time({"run": run, "scenes": chunk, "wall": wall, "jobs": [r.get("seconds") for r in results]})
    return failed

## 6. The plan's other scenes

Only with `ALL_SCENES = True` (step 1). SAM 3 first for every scene that needs it, then each run's tracking; finished
scenes are skipped, and a scene that fails is reported while the others go on. After a disconnect, run steps 2, 3, 4
and this one again.

In [ ]:
if not ALL_SCENES:
    print("ALL_SCENES is False: read step 5's checks, then set it to True in step 1 and run this cell again")
else:
    start, failed = time.time(), {}
    todo = [s for s in PLAN_SCENES if sam_missing(s)]
    for i, s in enumerate(todo, 1):
        try:
            ensure_sam3(s)
        except Exception as error:  # noqa: BLE001 - one scene's failure must not stop the others
            failed[f"sam3 {s}"] = f"{type(error).__name__}: {error}"
        print(f"SAM 3 {i}/{len(todo)} {s}: {(time.time() - start) / 60:.0f} min so far", flush=True)
    for run, scenes in PLAN.items():
        ready = [s for s in scenes if not sam_missing(s)]
        failed.update({f"{run} {k}": v for k, v in track_run(run, ready).items()})
        print(f"{run}: {sum(done(run, s) for s in scenes)} of {len(scenes)} scenes done, "
              f"{(time.time() - start) / 60:.0f} min so far", flush=True)
    print("failed:", failed or "none")

## 7. Status

In [ ]:
for r, scenes in PLAN.items():
    print(f"{r:22s} {sum(done(r, s) for s in scenes):2d} / {len(scenes)} scenes")
print(f"{'SAM 3':22s} {sum(not sam_missing(s) for s in PLAN_SCENES):2d} / {len(PLAN_SCENES)} scenes")

## Using the outputs locally

Download `MyDrive/mvp/rerun_detection/runs/<run>/` (e.g. to `data/rerun/<run>/`). The sheets are made from its tracks
and the dense depth in `data/vggt`, with the sizing every experiment so far used (`scene_sheet_creator.py` at
`50fe543`, as in `pipeline_fixes.md`), and scored against `data/scene_sheets_75` on the run's scenes:

```bash
E=experiments/2026-11-18-scene-sheet-formats
PYTHONPATH=src python $E/merge_points.py --sheets data/rerun/<run> --out data/rerun_sheets/<run> \
    --scenes data/rerun/<run>/scenes.txt --code old/src --no-merge
# fixed-prompt runs only: their renamed questions need gold sheets made again
python src/mvp/openvocab_adapter/rebuild_scene_sheets.py --root data/rerun_sheets/<run> --source gold \
    --scenes $(sed 's|/|__|' data/rerun/<run>/scenes.txt)
PYTHONPATH=src python $E/pipeline_scorecard.py --old data/scene_sheets_75 --new data/rerun_sheets/<run> \
    --scenes data/rerun/<run>/scenes.txt
```

`original_s020` against `data/scene_sheets_75` is what the lower threshold adds, together with the new SAM 3 run
(small, by step 5's check 3); `fixed_s020` against `original_s020` is what the prompt fixes add. Merging
(`merge_points.py` without `--no-merge`) can then be applied to a rebuilt folder.